In [1]:
import os
from pathlib import Path

import dotenv
import numpy as np
import pandas as pd

# Load environment variables
dotenv.load_dotenv()

# Set up output folder
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# 300: Prepare Scenarios for MAGICC Runs

This notebook prepares scenario batches for running climate projections with MAGICC.

## Overview

Starting from 1501 scenarios, we filter for those achieving net-zero CO2 (by 2090) and 
net-zero Kyoto GHG emissions, then create multiple batches with different MAGICC configurations.

## Batches Created

| Batch | Description | MAGICC Flag | Input File | Filter |
|-------|-------------|-------------|------------|--------|
| A | Original scenarios | ALL | Harmonised data | Net-zero CO2 ≤2090 |
| B | Original scenarios | CO2 | Harmonised data | Net-zero CO2 ≤2090 |
| C | Original scenarios | GHG | Harmonised data | Net-zero CO2 ≤2090 |
| D | Net-zero CO2 extended | CO2 | NZCO2 extended | Net-zero CO2 ≤2090 |
| D (ALL) | Net-zero CO2 extended | ALL | NZCO2 extended | Net-zero CO2 ≤2090 |
| D (GHG) | Net-zero CO2 extended | GHG | NZCO2 extended | Net-zero CO2 ≤2090 |
| E | Net-zero GHG extended | ALL | NZKyoto extended | Net-zero Kyoto GHG |
| E (GHG) | Net-zero GHG extended | GHG | NZKyoto extended | Net-zero Kyoto GHG |
| E (CO2) | Net-zero GHG extended | CO2 | NZKyoto extended | Net-zero Kyoto GHG |

## Inputs
- `102_netzero_timings.csv`: Net-zero timing metadata
- `101_harmonised_data.csv`: Harmonised emissions data
- `102_netzero_co2_extended.csv`: Emissions extended from net-zero CO2
- `102_netzero_kyoto_extended.csv`: Emissions extended from net-zero Kyoto GHG

## Output
- `300_all_batches_{VERSION}.csv`: Combined batch configuration file

## Constants

In [2]:
ENSEMBLE_MEMBERS = 600
INPUT_FILE = OUTPUT_FOLDER / "102_netzero_timings.csv"
HARMONISED_DATA_FILE = OUTPUT_FOLDER / "101_harmonised_data.csv"
NETZERO_EXTENDED_FILE = OUTPUT_FOLDER / "102_netzero_co2_extended.csv"
NETZERO_EXTENDED_FILE_GHG = OUTPUT_FOLDER / "102_netzero_kyoto_extended.csv"
VERSION = "v0.4"

In [3]:
df = pd.read_csv(INPUT_FILE)

## Load and Filter Data

Check the total number of scenarios in the input data.

In [4]:
n_total = len(df)
print(f"There are {n_total} scenarios")

There are 1501 scenarios


In [5]:
df

,model,scenario,year_netzero_co2,year_netzero_kyoto_ar6gwp100,achieve_netzero_co2,achieve_netzero_ghg
0,AIM/CGE 2.0,SSP1-19,2058.0,2073.0,True,True
1,AIM/CGE 2.0,SSP1-26,NaN,NaN,False,False
2,AIM/CGE 2.0,SSP1-34,NaN,NaN,False,False
3,AIM/CGE 2.0,SSP1-45,NaN,NaN,False,False
4,AIM/CGE 2.0,SSP1-Baseline,NaN,NaN,False,False
...,...,...,...,...,...,...
1496,WITCH-GLOBIOM 4.4,CD-LINKS-NPi,NaN,NaN,False,False
1497,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1000,2079.0,2093.0,True,True
1498,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1600,2098.0,NaN,True,False
1499,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_400,2059.0,2071.0,True,True


Filter for scenarios that achieve net-zero CO2 emissions and set the model-scenario index.

In [6]:
df_for_initial_batches = (
    df[df["achieve_netzero_co2"]]
    .set_index(["model", "scenario"])
)

In [7]:
orig_scens = len(df_for_initial_batches)

In [8]:
omit_due_to_late_nz = df_for_initial_batches["year_netzero_co2"][
    df_for_initial_batches["year_netzero_co2"]>2090
]
len(omit_due_to_late_nz)

58

In [9]:
df_for_initial_batches = df_for_initial_batches[
    df_for_initial_batches["year_netzero_co2"] <= 2090
]
print(
    f"Original: {orig_scens}, Omitted: {len(omit_due_to_late_nz)}, Now considered: {orig_scens - len(omit_due_to_late_nz)}"
)

Original: 743, Omitted: 58, Now considered: 685


## Create Batch Dataframes

### Batch A: Original scenarios with all forcers

In [10]:
df_batch_a = (
    df_for_initial_batches
    .copy()
    .assign(
        ENSEMBLE_MEMBERS=ENSEMBLE_MEMBERS,
        FILE=HARMONISED_DATA_FILE,
        MAGICC_FLAG="ALL"
    )
    .reset_index()
    .rename(columns={"model": "MODEL", "scenario": "SCENARIO"})
    .drop(columns=[
        "year_netzero_co2",
        "year_netzero_kyoto_ar6gwp100",
        "achieve_netzero_co2",
        "achieve_netzero_ghg"
    ])
)

### Batch B: Original scenarios with CO2 only

In [11]:
df_batch_b = df_batch_a.assign(MAGICC_FLAG="CO2")

### Batch C: Original scenarios with GHG only

In [12]:
df_batch_c = df_batch_a.assign(MAGICC_FLAG="GHG")

### Batch D: Net-zero CO2 extended scenarios with CO2 only

In [13]:
df_batch_d = df_batch_b.assign(FILE=NETZERO_EXTENDED_FILE)

In [14]:
df_batch_d["SCENARIO"] = df_batch_d["SCENARIO"] + "_NZCO2"
df_batch_d

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE,MAGICC_FLAG
0,AIM/CGE 2.0,SSP1-19_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2
1,AIM/CGE 2.0,SSP2-19_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2
2,AIM/CGE 2.0,SSP2-26_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2
3,AIM/CGE 2.0,SSP3-34_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2
4,AIM/CGE 2.0,SSP4-26_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2
...,...,...,...,...,...
680,WITCH-GLOBIOM 3.1,SSP4-19_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2
681,WITCH-GLOBIOM 4.4,CD-LINKS-INDC2030i_1600_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2
682,WITCH-GLOBIOM 4.4,CD-LINKS-NDC2030i_1000_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2
683,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1000_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2


### Batch D (ALL): Net-zero CO2 extended scenarios with all forcers

In [15]:
df_batch_d_all = df_batch_d.assign(MAGICC_FLAG="ALL")

In [16]:
df_batch_d_all.head()

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE,MAGICC_FLAG
0,AIM/CGE 2.0,SSP1-19_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
1,AIM/CGE 2.0,SSP2-19_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
2,AIM/CGE 2.0,SSP2-26_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
3,AIM/CGE 2.0,SSP3-34_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
4,AIM/CGE 2.0,SSP4-26_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL


In [17]:
df_batch_d_ghg = df_batch_d.assign(MAGICC_FLAG="GHG")

In [18]:
df_batch_d_ghg.head()

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE,MAGICC_FLAG
0,AIM/CGE 2.0,SSP1-19_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,GHG
1,AIM/CGE 2.0,SSP2-19_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,GHG
2,AIM/CGE 2.0,SSP2-26_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,GHG
3,AIM/CGE 2.0,SSP3-34_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,GHG
4,AIM/CGE 2.0,SSP4-26_NZCO2,600,/Users/ganti/Documents/code/ar7_netzero_assess...,GHG


### Batch E: Net-zero GHG extended scenarios with all forcers

This batch uses a different filter (achieve_netzero_ghg) and the GHG extended file.

In [19]:
df_batch_e = (
    df[df["achieve_netzero_ghg"]]
    .copy()
    .assign(
        ENSEMBLE_MEMBERS=ENSEMBLE_MEMBERS,
        FILE=NETZERO_EXTENDED_FILE_GHG,
        MAGICC_FLAG="ALL"
    )
    .rename(columns={"model": "MODEL", "scenario": "SCENARIO"})
    .drop(columns=[
        "year_netzero_co2",
        "year_netzero_kyoto_ar6gwp100",
        "achieve_netzero_co2",
        "achieve_netzero_ghg"
    ])
)

df_batch_e

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE,MAGICC_FLAG
0,AIM/CGE 2.0,SSP1-19,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
5,AIM/CGE 2.0,SSP2-19,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
29,AIM/CGE 2.1,CD-LINKS-NPi2020_400,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
34,AIM/CGE V2.2,ENGAGE-Feasibility-1000/Institutional,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
40,AIM/CGE V2.2,ENGAGE-Feasibility-Maximum-Effort/Institutional,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
...,...,...,...,...,...
1460,WITCH-GLOBIOM 3.1,SSP1-19,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
1474,WITCH-GLOBIOM 3.1,SSP4-19,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
1495,WITCH-GLOBIOM 4.4,CD-LINKS-NDC2030i_1000,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
1497,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1000,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL


In [20]:
df_batch_e["SCENARIO"] = df_batch_e["SCENARIO"] + "_NZKyoto"
df_batch_e

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE,MAGICC_FLAG
0,AIM/CGE 2.0,SSP1-19_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
5,AIM/CGE 2.0,SSP2-19_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
29,AIM/CGE 2.1,CD-LINKS-NPi2020_400_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
34,AIM/CGE V2.2,ENGAGE-Feasibility-1000/Institutional_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
40,AIM/CGE V2.2,ENGAGE-Feasibility-Maximum-Effort/Institutiona...,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
...,...,...,...,...,...
1460,WITCH-GLOBIOM 3.1,SSP1-19_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
1474,WITCH-GLOBIOM 3.1,SSP4-19_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
1495,WITCH-GLOBIOM 4.4,CD-LINKS-NDC2030i_1000_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
1497,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1000_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL


In [21]:
df_batch_e_ghg = df_batch_e.assign(MAGICC_FLAG="GHG")
df_batch_e_ghg.head()

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE,MAGICC_FLAG
0,AIM/CGE 2.0,SSP1-19_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,GHG
5,AIM/CGE 2.0,SSP2-19_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,GHG
29,AIM/CGE 2.1,CD-LINKS-NPi2020_400_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,GHG
34,AIM/CGE V2.2,ENGAGE-Feasibility-1000/Institutional_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,GHG
40,AIM/CGE V2.2,ENGAGE-Feasibility-Maximum-Effort/Institutiona...,600,/Users/ganti/Documents/code/ar7_netzero_assess...,GHG


In [22]:
df_batch_e_co2 = df_batch_e.assign(MAGICC_FLAG="CO2")
df_batch_e_co2.head()

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE,MAGICC_FLAG
0,AIM/CGE 2.0,SSP1-19_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2
5,AIM/CGE 2.0,SSP2-19_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2
29,AIM/CGE 2.1,CD-LINKS-NPi2020_400_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2
34,AIM/CGE V2.2,ENGAGE-Feasibility-1000/Institutional_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2
40,AIM/CGE V2.2,ENGAGE-Feasibility-Maximum-Effort/Institutiona...,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2


## Concatenate All Batches

In [23]:
df_all_batches = pd.concat(
    [
        df_batch_a, 
        df_batch_b, 
        df_batch_c, 
        df_batch_d, 
        df_batch_d_all,
        df_batch_d_ghg,
        df_batch_e,
        df_batch_e_ghg,
        df_batch_e_co2
    ],
    ignore_index=True,
)

df_all_batches

,MODEL,SCENARIO,ENSEMBLE_MEMBERS,FILE,MAGICC_FLAG
0,AIM/CGE 2.0,SSP1-19,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
1,AIM/CGE 2.0,SSP2-19,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
2,AIM/CGE 2.0,SSP2-26,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
3,AIM/CGE 2.0,SSP3-34,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
4,AIM/CGE 2.0,SSP4-26,600,/Users/ganti/Documents/code/ar7_netzero_assess...,ALL
...,...,...,...,...,...
5089,WITCH-GLOBIOM 3.1,SSP1-19_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2
5090,WITCH-GLOBIOM 3.1,SSP4-19_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2
5091,WITCH-GLOBIOM 4.4,CD-LINKS-NDC2030i_1000_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2
5092,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_1000_NZKyoto,600,/Users/ganti/Documents/code/ar7_netzero_assess...,CO2


In [24]:
df_all_batches.to_csv(OUTPUT_FOLDER / f"300_all_batches_{VERSION}.csv")